# Building the London Intelligence Dataset

## Objective

Build a multi-layer London property intelligence dataset by integrating validated and prepared urban datasets with the property market dataset.

The objective is to answer the following question:

> **How can multiple urban intelligence datasets be combined to support better property investment decisions?**

The final dataset will combine housing market performance with wider neighbourhood characteristics to create a comprehensive view of London boroughs.

This integrated dataset will support:

* property investment analysis
* neighbourhood comparison
* feature engineering
* predictive modelling
* data-driven decision-making

---

## Data Integration Philosophy

Each dataset layer is prepared and validated independently before integration.

This notebook focuses on:

* loading prepared analytical datasets
* verifying integration compatibility
* checking merge keys and data granularity
* combining multiple intelligence layers
* validating the final integrated dataset

---

## Data Architecture

The final London Intelligence Dataset will integrate the following data layers:

### 1. Property Market Layer

**Source**

HM Land Registry Price Paid Data

**Prepared dataset**

`borough_year_features.csv`

**Contains**

* Transaction activity
* Average property price
* Median property price
* Minimum and maximum prices
* Price volatility
* Historical price growth
* Prediction targets

---

### 2. Transport Accessibility Layer

**Source**

PTAL Data

**Purpose**

Measure transport accessibility and connectivity across London boroughs.

---

### 3. Crime Layer

**Source**

Metropolitan Police Crime Data

**Purpose**

Represent neighbourhood safety using historical crime indicators.

---

### 4. Income Layer

**Source**

HMRC Income Data

**Purpose**

Represent borough-level economic characteristics and purchasing power.

---

### 5. Deprivation Layer

**Source**

Index of Multiple Deprivation (IMD)

**Purpose**

Measure socio-economic conditions across London boroughs.

---

### 6. Population Layer

**Source**

ONS Population Data

**Prepared dataset**

`london_population.csv`

**Purpose**

Represent demographic characteristics, including population size and population density.

---

## Workflow

Prepared analytical datasets

↓

Load one intelligence layer

↓

Audit structure, granularity and merge compatibility

↓

Validate merge keys

↓

Integrate into the London Intelligence Dataset

↓

Perform final data quality validation

↓

Repeat for the next data layer

↓

Save the final London Intelligence Dataset


## Population Layer Integration Audit

The population layer has already been cleaned, standardised and validated during the data preparation stage.

Before integration, this notebook verifies that the prepared population dataset is compatible with the borough-year analytical dataset.

This integration audit answers the following questions:

* Is the dataset at the correct level of granularity?
* Which columns can be used as merge keys?
* Does the time coverage match the analytical dataset?
* Are there any missing values?
* Are there duplicate borough-year records?
* Is the dataset ready to be integrated into the London Intelligence Dataset?


In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# Project folders

RAW_DATA = Path("../data/raw")
PROCESSED_DATA = Path("../data/processed")

print("Raw folder:", RAW_DATA)
print("Processed folder:", PROCESSED_DATA)

Raw folder: ..\data\raw
Processed folder: ..\data\processed


In [3]:
borough_year_df = pd.read_csv(
    PROCESSED_DATA / "borough_year_features.csv"
)

print(borough_year_df.shape)
borough_year_df.head()

(264, 12)


,District,Year,Transactions,Average_Price,Median_Price,Min_Price,Max_Price,Price_STD,Average_Price_Growth,Median_Price_Growth,Target_Average_Price_Growth,Target_Median_Price_Growth
0,BARKING AND DAGENHAM,2018,2404,371419.146007,306750.0,100,20495000,9.097848e+05,NaN,NaN,0.059790,0.010595
1,BARKING AND DAGENHAM,2019,2146,393626.402144,310000.0,100,23160000,1.059141e+06,0.059790,0.010595,0.189852,0.032258
2,BARKING AND DAGENHAM,2020,1717,468356.971462,320000.0,100,41750000,1.792652e+06,0.189852,0.032258,-0.099888,0.046875
3,BARKING AND DAGENHAM,2021,2537,421573.503745,335000.0,100,50920000,1.783341e+06,-0.099888,0.046875,0.192617,0.104478
4,BARKING AND DAGENHAM,2022,2110,502775.688152,370000.0,100,98800000,2.466056e+06,0.192617,0.104478,-0.129364,0.010811


In [4]:
# Load the prepared population layer

population_df = pd.read_csv(
    PROCESSED_DATA / "london_population.csv"
)

print(population_df.shape)

population_df.head()

(396, 5)


,Code,Name,Geography,Year,Population
0,E09000007,Camden,London Borough,2022,217365
1,E09000001,City of London,London Borough,2022,11457
2,E09000012,Hackney,London Borough,2022,261632
3,E09000013,Hammersmith and Fulham,London Borough,2022,185506
4,E09000014,Haringey,London Borough,2022,262413


In [5]:
population_df.shape

(396, 5)

In [6]:
population_df.columns

Index(['Code', 'Name', 'Geography', 'Year', 'Population'], dtype='object')

# Data quality audit
# Verify the structure, completeness and merge compatibility
# of the population dataset before integration.

In [7]:
# Integration audit
# Verify that the prepared population layer is ready
# to be merged with the borough-year analytical dataset.

print("=" * 60)
print("POPULATION LAYER INTEGRATION AUDIT")
print("=" * 60)

print(f"Dataset shape: {population_df.shape}")

print("\nColumns:")
print(population_df.columns.tolist())

print("\nData types:")
print(population_df.dtypes)

print("\nMissing values:")
print(population_df.isna().sum())

print(f"\nNumber of boroughs: {population_df['Name'].nunique()}")

print("\nYears covered:")
print(sorted(population_df["Year"].unique()))

print("\nDuplicate borough-year records:")
print(
    population_df.duplicated(
        subset=["Name", "Year"]
    ).sum()
)

POPULATION LAYER INTEGRATION AUDIT
Dataset shape: (396, 5)

Columns:
['Code', 'Name', 'Geography', 'Year', 'Population']

Data types:
Code          object
Name          object
Geography     object
Year           int64
Population     int64
dtype: object

Missing values:
Code          0
Name          0
Geography     0
Year          0
Population    0
dtype: int64

Number of boroughs: 33

Years covered:
[np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]

Duplicate borough-year records:
0


## Merge Compatibility Check

Before merging the population layer with the borough-year analytical dataset, both datasets are compared to verify that they share compatible merge keys and the same analytical coverage.

This validation answers the following questions:

* Do both datasets cover the same London boroughs?
* Do both datasets cover the same years?
* Are the merge key values consistent?
* Are any boroughs or years missing from either dataset?
* Is the population layer ready to be merged without data loss?


In [8]:
# Compare the analytical coverage of both datasets

print("=" * 60)
print("ANALYTICAL COVERAGE")
print("=" * 60)

print(f"Property dataset shape   : {borough_year_df.shape}")
print(f"Population dataset shape : {population_df.shape}")

print(f"\nProperty boroughs   : {borough_year_df['District'].nunique()}")
print(f"Population boroughs : {population_df['Name'].nunique()}")

print(f"\nProperty years   : {sorted(borough_year_df['Year'].unique())}")
print(f"Population years : {sorted(population_df['Year'].unique())}")

ANALYTICAL COVERAGE
Property dataset shape   : (264, 12)
Population dataset shape : (396, 5)

Property boroughs   : 33
Population boroughs : 33

Property years   : [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Population years : [np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]


In [9]:
# Compare borough names between datasets

property_boroughs = set(borough_year_df["District"])

population_boroughs = set(population_df["Name"])

print("Only in property dataset:")
print(sorted(property_boroughs - population_boroughs))

print("\nOnly in population dataset:")
print(sorted(population_boroughs - property_boroughs))

Only in property dataset:
['BARKING AND DAGENHAM', 'BARNET', 'BEXLEY', 'BRENT', 'BROMLEY', 'CAMDEN', 'CITY OF LONDON', 'CITY OF WESTMINSTER', 'CROYDON', 'EALING', 'ENFIELD', 'GREENWICH', 'HACKNEY', 'HAMMERSMITH AND FULHAM', 'HARINGEY', 'HARROW', 'HAVERING', 'HILLINGDON', 'HOUNSLOW', 'ISLINGTON', 'KENSINGTON AND CHELSEA', 'KINGSTON UPON THAMES', 'LAMBETH', 'LEWISHAM', 'MERTON', 'NEWHAM', 'REDBRIDGE', 'RICHMOND UPON THAMES', 'SOUTHWARK', 'SUTTON', 'TOWER HAMLETS', 'WALTHAM FOREST', 'WANDSWORTH']

Only in population dataset:
['Barking and Dagenham', 'Barnet', 'Bexley', 'Brent', 'Bromley', 'Camden', 'City of London', 'Croydon', 'Ealing', 'Enfield', 'Greenwich', 'Hackney', 'Hammersmith and Fulham', 'Haringey', 'Harrow', 'Havering', 'Hillingdon', 'Hounslow', 'Islington', 'Kensington and Chelsea', 'Kingston upon Thames', 'Lambeth', 'Lewisham', 'Merton', 'Newham', 'Redbridge', 'Richmond upon Thames', 'Southwark', 'Sutton', 'Tower Hamlets', 'Waltham Forest', 'Wandsworth', 'Westminster']


## Merge Key Standardisation

Although both datasets describe the same London boroughs, their borough names are not stored in a consistent format.

To ensure reliable dataset integration, a standard merge key is created for each dataset.

The merge key:

* converts borough names to uppercase
* removes unnecessary whitespace
* preserves the original borough names
* provides a consistent key for all future dataset integrations


In [10]:
# Create standard merge keys for borough matching

borough_year_df["Merge_Key"] = (
    borough_year_df["District"]
    .str.upper()
    .str.strip()
)

population_df["Merge_Key"] = (
    population_df["Name"]
    .str.upper()
    .str.strip()
)

print(borough_year_df["Merge_Key"].nunique())
print(population_df["Merge_Key"].nunique())

33
33


In [11]:
property_keys = set(borough_year_df["Merge_Key"])

population_keys = set(population_df["Merge_Key"])

print("Only in property:")
print(sorted(property_keys - population_keys))

print("\nOnly in population:")
print(sorted(population_keys - property_keys))

Only in property:
['CITY OF WESTMINSTER']

Only in population:
['WESTMINSTER']


## Borough Name Harmonisation

After standardising the merge keys, any remaining naming differences are identified and resolved.

Known borough naming variations are harmonised using explicit mapping rules to ensure consistent matching across all datasets while preserving the original source data.


In [12]:
# Standardise known borough naming differences

borough_mapping = {
    "WESTMINSTER": "CITY OF WESTMINSTER"
}

population_df["Merge_Key"] = (
    population_df["Merge_Key"]
    .replace(borough_mapping)
)

print(population_df["Merge_Key"].nunique())

33


In [13]:
property_keys = set(borough_year_df["Merge_Key"])

population_keys = set(population_df["Merge_Key"])

print("Only in property:")
print(sorted(property_keys - population_keys))

print("\nOnly in population:")
print(sorted(population_keys - property_keys))

Only in property:
[]

Only in population:
[]


## Merge Validation

After standardising the merge keys and harmonising known naming differences, the compatibility of the property and population datasets is verified.

The validation confirms that:

* all London boroughs are successfully matched
* no unmatched merge keys remain
* the datasets are ready for integration


In [14]:
# Standardise known borough naming differences

borough_mapping = {
    "WESTMINSTER": "CITY OF WESTMINSTER"
}

population_df["Merge_Key"] = (
    population_df["Merge_Key"]
    .replace(borough_mapping)
)

In [15]:
property_keys = set(borough_year_df["Merge_Key"])
population_keys = set(population_df["Merge_Key"])

print("Only in property:")
print(sorted(property_keys - population_keys))

print("\nOnly in population:")
print(sorted(population_keys - property_keys))

Only in property:
[]

Only in population:
[]


## Final Dataset Schema

Before integrating the population layer, the structure of the final analytical dataset is defined.

The integrated dataset will retain only variables that provide analytical value or support future dataset integration.

Temporary integration columns and duplicated variables will be removed after the merge to keep the dataset clean, interpretable and suitable for downstream analysis.


In [16]:
# Final dataset schema

columns_to_keep = [
    "District",
    "Code",
    "Year",
    "Transactions",
    "Average_Price",
    "Median_Price",
    "Min_Price",
    "Max_Price",
    "Price_STD",
    "Average_Price_Growth",
    "Median_Price_Growth",
    "Target_Average_Price_Growth",
    "Target_Median_Price_Growth",
    "Population"
]

temporary_columns = [
    "Merge_Key",
    "Name",
    "Geography"
]

print("Columns to keep:")
print(columns_to_keep)

print("\nTemporary integration columns:")
print(temporary_columns)

Columns to keep:
['District', 'Code', 'Year', 'Transactions', 'Average_Price', 'Median_Price', 'Min_Price', 'Max_Price', 'Price_STD', 'Average_Price_Growth', 'Median_Price_Growth', 'Target_Average_Price_Growth', 'Target_Median_Price_Growth', 'Population']

Temporary integration columns:
['Merge_Key', 'Name', 'Geography']


## Merge Strategy

The property dataset is treated as the anchor dataset for the London Intelligence Dataset.

A left join is used to preserve every borough-year record in the property dataset while enriching it with population information.

If no matching population record exists, the population variables remain missing (`NaN`). No imputation is performed during the integration stage.


## Merge Population Layer

The validated population layer is integrated into the borough-year analytical dataset using the standard merge key.

A left join is performed to preserve all property records while adding borough-level population information.

After the merge, temporary integration columns and duplicated variables will be removed to produce a clean analytical dataset.


In [17]:
# Merge population layer using borough-year grain

london_intelligence_df = borough_year_df.merge(
    population_df[
        [
            "Merge_Key",
            "Year",
            "Code",
            "Population"
        ]
    ],
    on=["Merge_Key", "Year"],
    how="left"
)

print(london_intelligence_df.shape)

london_intelligence_df.head()

(264, 15)


,District,Year,Transactions,Average_Price,Median_Price,Min_Price,Max_Price,Price_STD,Average_Price_Growth,Median_Price_Growth,Target_Average_Price_Growth,Target_Median_Price_Growth,Merge_Key,Code,Population
0,BARKING AND DAGENHAM,2018,2404,371419.146007,306750.0,100,20495000,9.097848e+05,NaN,NaN,0.059790,0.010595,BARKING AND DAGENHAM,E09000002,216298.0
1,BARKING AND DAGENHAM,2019,2146,393626.402144,310000.0,100,23160000,1.059141e+06,0.059790,0.010595,0.189852,0.032258,BARKING AND DAGENHAM,E09000002,218828.0
2,BARKING AND DAGENHAM,2020,1717,468356.971462,320000.0,100,41750000,1.792652e+06,0.189852,0.032258,-0.099888,0.046875,BARKING AND DAGENHAM,E09000002,219227.0
3,BARKING AND DAGENHAM,2021,2537,421573.503745,335000.0,100,50920000,1.783341e+06,-0.099888,0.046875,0.192617,0.104478,BARKING AND DAGENHAM,E09000002,218714.0
4,BARKING AND DAGENHAM,2022,2110,502775.688152,370000.0,100,98800000,2.466056e+06,0.192617,0.104478,-0.129364,0.010811,BARKING AND DAGENHAM,E09000002,220039.0


In [18]:
london_intelligence_df.columns.tolist()

['District',
 'Year',
 'Transactions',
 'Average_Price',
 'Median_Price',
 'Min_Price',
 'Max_Price',
 'Price_STD',
 'Average_Price_Growth',
 'Median_Price_Growth',
 'Target_Average_Price_Growth',
 'Target_Median_Price_Growth',
 'Merge_Key',
 'Code',
 'Population']

## Post-Merge Validation

After integrating the population layer, the merged dataset is validated to ensure that the integration has preserved the analytical structure of the dataset.

The validation confirms:

* the expected number of records is retained
* the merge keys remain unique
* population values have been added correctly
* missing values are consistent with the available population data
* the integrated dataset is ready for further enrichment


In [19]:
# Validate the overall structure of the merged dataset

print("=" * 60)
print("DATASET INTEGRITY")
print("=" * 60)

print(f"Dataset shape: {london_intelligence_df.shape}")

print(f"Number of boroughs: {london_intelligence_df['District'].nunique()}")

print(f"Years covered: {sorted(london_intelligence_df['Year'].unique())}")

DATASET INTEGRITY
Dataset shape: (264, 15)
Number of boroughs: 33
Years covered: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


In [20]:
# Validate borough-year uniqueness

duplicates = london_intelligence_df.duplicated(
    subset=["District", "Year"]
).sum()

print(f"Duplicate Borough-Year records: {duplicates}")

Duplicate Borough-Year records: 0


In [21]:
# Check population availability

print(london_intelligence_df["Population"].isna().sum())

99


## Feature Validation

The newly integrated population feature is inspected to confirm that values have been successfully added and that missing values occur only where population data is unavailable.

This final validation confirms that the population layer has been integrated correctly before the dataset is saved.


In [22]:
# Inspect the integrated population feature

print(london_intelligence_df["Population"].describe())

count       165.000000
mean     268266.048485
std       77646.613109
min        8165.000000
25%      217050.000000
50%      279527.000000
75%      327426.000000
max      392610.000000
Name: Population, dtype: float64


In [23]:
# Remove temporary integration columns

london_intelligence_df = london_intelligence_df.drop(
    columns=["Merge_Key"]
)

london_intelligence_df.head()

,District,Year,Transactions,Average_Price,Median_Price,Min_Price,Max_Price,Price_STD,Average_Price_Growth,Median_Price_Growth,Target_Average_Price_Growth,Target_Median_Price_Growth,Code,Population
0,BARKING AND DAGENHAM,2018,2404,371419.146007,306750.0,100,20495000,9.097848e+05,NaN,NaN,0.059790,0.010595,E09000002,216298.0
1,BARKING AND DAGENHAM,2019,2146,393626.402144,310000.0,100,23160000,1.059141e+06,0.059790,0.010595,0.189852,0.032258,E09000002,218828.0
2,BARKING AND DAGENHAM,2020,1717,468356.971462,320000.0,100,41750000,1.792652e+06,0.189852,0.032258,-0.099888,0.046875,E09000002,219227.0
3,BARKING AND DAGENHAM,2021,2537,421573.503745,335000.0,100,50920000,1.783341e+06,-0.099888,0.046875,0.192617,0.104478,E09000002,218714.0
4,BARKING AND DAGENHAM,2022,2110,502775.688152,370000.0,100,98800000,2.466056e+06,0.192617,0.104478,-0.129364,0.010811,E09000002,220039.0


In [24]:
# Save the London Intelligence Dataset

london_intelligence_df.to_csv(
    PROCESSED_DATA / "london_intelligence_dataset.csv",
    index=False
)

print("London Intelligence Dataset saved successfully.")

London Intelligence Dataset saved successfully.


## Population Coverage

The availability of population values is assessed after integration.

This validation distinguishes between expected missing values caused by unavailable source data and unexpected missing values that could indicate integration problems.

The objective is to confirm that population information has been successfully integrated for all years covered by the source dataset.

In [25]:
# Population coverage summary

matched = london_intelligence_df["Population"].notna().sum()
missing = london_intelligence_df["Population"].isna().sum()
total = len(london_intelligence_df)

print("=" * 60)
print("POPULATION COVERAGE")
print("=" * 60)

print(f"Population available : {matched} / {total} ({matched / total:.1%})")
print(f"Missing values       : {missing} / {total} ({missing / total:.1%})")

POPULATION COVERAGE
Population available : 165 / 264 (62.5%)
Missing values       : 99 / 264 (37.5%)


In [26]:
# Validate population coverage for years where data is expected

available_years = london_intelligence_df["Year"] <= 2022

missing_available_years = (
    london_intelligence_df.loc[
        available_years,
        "Population"
    ]
    .isna()
    .sum()
)

print(
    f"Missing population values (2018–2022): {missing_available_years}"
)

Missing population values (2018–2022): 0


### Interpretation

Population values are available for every borough-year record covered by the ONS population dataset (2018–2022).

The remaining missing values correspond to years 2023–2025, for which official population estimates are not yet available.

These missing values are therefore expected and do not indicate an integration error.

## Merge Integrity Check

The merged dataset is verified to ensure that the integration process has preserved the expected analytical structure.

Assertions are used to detect unexpected changes in future executions of this notebook.

In [27]:
# Validate merge integrity

assert london_intelligence_df.shape[0] == borough_year_df.shape[0]

print("Merge integrity check passed.")

Merge integrity check passed.


## Borough Code Validation

The official ONS borough codes are validated to ensure that each borough is associated with a single unique identifier.

In [28]:
# Validate borough codes

borough_codes = (
    london_intelligence_df.groupby("District")["Code"]
    .nunique()
)

assert borough_codes.max() == 1

print("All borough codes are unique.")

All borough codes are unique.


In [29]:
print(london_intelligence_df["Population"].describe())

count       165.000000
mean     268266.048485
std       77646.613109
min        8165.000000
25%      217050.000000
50%      279527.000000
75%      327426.000000
max      392610.000000
Name: Population, dtype: float64


## Population Layer Integration Complete

The Population Layer has been successfully integrated into the London Intelligence Dataset.

Integration results:

- All 33 London boroughs were matched successfully.
- Borough-year granularity was preserved.
- No duplicate records were introduced.
- Population data is complete for all available years (2018–2022).
- Missing values occur only for years where official population estimates are unavailable (2023–2025).
- The London Intelligence Dataset is now ready for integration of the next urban intelligence layer.